# 🐼 Séance 3 — Pandas : charger, inspecter, nettoyer · Partie 1 : pratiquer
_Master IA‑GI — Python pour les sciences de données_

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelallioui/python-for-ds-s03/blob/main/tp-seance-03.ipynb)

**Comment ouvrir ce notebook :** sur votre ordinateur, ouvrez-le **depuis son dossier** `seance-03/`. Il est **indépendant** : il n'a besoin d'aucun fichier fourni ; la partie B crée un dossier `fichiers_tp/` à côté de lui pour ses fichiers d'essai. Pandas doit être installé (`pip install pandas` ; déjà présent avec Anaconda). Dans Colab, le bouton ci-dessus suffit : il ouvre le notebook dans **votre** compte Google ; pour **garder votre travail**, faites **Fichier ▸ Enregistrer une copie dans Drive**.

**Objectifs** — à la fin de ce notebook, vous saurez :
- situer Pandas, **démarrer** avec lui, distinguer une **Series** d'un **DataFrame** et comprendre l'**alignement par étiquette**
- **charger** un fichier avec `read_csv` (même abîmé), le **regarder** (`head`, `shape`, `info`, `describe`) et **écrire** un fichier avec `to_csv`
- **sélectionner** des colonnes et des lignes avec `[]`, `loc`, `iloc` et des **masques** combinés
- **diagnostiquer** un jeu : valeurs manquantes, doublons, libellés incohérents
- **nettoyer en décidant** : supprimer, remplir, marquer ou laisser, **corriger** libellés et types, puis **créer**, renommer et trier des colonnes

**Durée estimée** : 2 heures environ · **Parties** :
- **A** · Rencontrer Pandas
- **B** · Charger, regarder, écrire
- **C** · Sélectionner
- **D** · Diagnostiquer
- **E** · Nettoyer

**Comment travailler seul**
1. Exécutez les cellules **dans l'ordre** (`Maj + Entrée`).
2. Lisez le *cours en bref*, puis **exécutez le code complet** : c'est lui qui contient les détails des diapos.
3. Faites chaque exercice en remplaçant les `...` par votre code, puis exécutez la cellule : un ✅ ou un ❌ s'affiche.
4. Un ❌ ? Ouvrez l'**indice 1**, puis 2, puis 3.

### 🧰 Pour commencer : importer, puis se contrôler
Exécutez la cellule ci-dessous. Elle importe ce qu'il faut et définit **`verifier`**, la fonction qui affiche ✅ ou ❌ sous chaque exercice. Rien de nouveau dans son écriture : une fonction (`def`), un `try` / `except` et un `lambda`, vus en séance 1.

In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd

print("Pandas", pd.__version__, "· NumPy", np.__version__)

def verifier(ex, description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (même si le test provoque une erreur)."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + ex + " · " + description)

### 🪪 Qui êtes-vous ?
Écrivez votre **nom** et votre **prénom** **entre guillemets**, en remplaçant les `...` (par exemple `nom = "Benali"`). Ces deux lignes me permettent de reconnaître votre travail : le contrôle reste ❌ tant qu'elles ne sont pas remplies. **Votre dépôt est public : n'y écrivez jamais de donnée personnelle (CNE, mot de passe, téléphone).**

In [ ]:
nom = ...
prenom = ...

print(nom, prenom)
verifier("identité", "nom et prénom renseignés", lambda: type(nom) is str and type(prenom) is str and len(nom.strip()) >= 2 and len(prenom.strip()) >= 2)

---
# Partie 1 — Pratiquer : de votre première Series au nettoyage
_Environ 2 heures · cinq parties (A à E) · 26 exercices directs._

Chaque exercice pose une **question directe** : vous écrivez le code, vous **affichez le résultat avec `print`**, et un ✅ ou un ❌ vous dit si c'est juste. Gardez le réflexe : **en cas de doute, `print` !**

Cette partie est **indépendante du Titanic** : elle travaille sur de petits tableaux inventés, dont un fichier de **20 inscrits** d'une faculté, plein de défauts exprès. Le Titanic vous attend dans le **mini-projet**, avec les mêmes gestes.

---
# Partie A · Rencontrer Pandas ⭐
_Diapos 7 à 11._

**Cours en bref**
- **Pandas** : créé en 2008 par Wes McKinney, **libre** (licence BSD), écrit en **Python, Cython et C** ; le code source est sur `github.com/pandas-dev/pandas`. Il repose sur **NumPy** et sert de point d'entrée aux autres bibliothèques.
- Deux objets centraux : la **Series** (une colonne : des valeurs et un **index**) et le **DataFrame** (un tableau : des Series qui partagent le même index).
- Pandas **aligne par étiquette** : `s1 + s2` additionne les valeurs qui ont la même étiquette ; sans vis-à-vis, le résultat est **NaN**. Une colonne affectée à un DataFrame se range de la même façon, **face à son étiquette**.
- **Bien démarrer** : `pip install pandas` (déjà fait dans Colab en général), `import pandas as pd`, `pd.__version__` (c'est fait dans la première cellule).
- **Sept portes d'entrée** : `pd.Series(liste)`, `pd.DataFrame(dictionnaire)`, `pd.DataFrame(liste de dictionnaires)`, `pd.read_csv`, `pd.read_excel`, `pd.read_json`, `pd.read_sql` (nommé seulement). Chaque lecture `read_...` a son écriture `to_...`.

In [ ]:
# Une Series : des valeurs, un index (les étiquettes) et un type
immatriculations = pd.Series([15, 20, 57], index=["Fes", "Meknes", "Khouribga"])
print(immatriculations)
print("valeur de Fes :", immatriculations["Fes"])

# Un DataFrame depuis un dictionnaire : une clé par colonne
etudiants = pd.DataFrame({"Nom": ["Ali", "Sara", "Driss"], "Age": [21, 23, 20], "Note": [14.5, 16.0, 12.0]})
print(etudiants)
print("chaque colonne est une Series :", type(etudiants["Note"]).__name__)
print("la moyenne des notes :", etudiants["Note"].mean())

In [ ]:
# Les étiquettes alignent les valeurs : b et c existent des deux côtés, a et d d'un seul
s1 = pd.Series([10, 20, 30], index=["a", "b", "c"])
s2 = pd.Series([100, 200, 300], index=["b", "c", "d"])
print(s1 + s2)
print("sous la Series, un tableau NumPy :", s1.to_numpy())

**✏️ Exercice A1 — Créer une Series ⭐**

Avec les valeurs `[15, 20, 57]` (immatriculations) et les étiquettes `["Fes", "Meknes", "Khouribga"]` :
1. `immat` : la **Series** correspondante
2. `valeur_khouribga` : la valeur de l'étiquette `"Khouribga"`

<details>
<summary>💡 Indice 1 · la piste</summary>

`pd.Series(...)` prend les valeurs, puis `index=` pour les étiquettes.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.Series([15, 20, 57], index=[...])` ; une valeur se lit avec des crochets et son étiquette.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`immat = pd.Series([15, 20, 57], index=["Fes", "Meknes", "Khouribga"])` ; `valeur_khouribga = immat["Khouribga"]`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
immat = ...
valeur_khouribga = ...

verifier("A1", "immat est une Series de 3 valeurs", lambda: isinstance(immat, pd.Series) and len(immat) == 3)
verifier("A1", "les étiquettes sont les trois villes", lambda: list(immat.index) == ["Fes", "Meknes", "Khouribga"])
verifier("A1", "Khouribga vaut 57", lambda: valeur_khouribga == 57)

**✏️ Exercice A2 — Créer un DataFrame depuis un dictionnaire ⭐**

1. `promo` : un **DataFrame** de 3 colonnes `Nom` (`"Ali"`, `"Sara"`, `"Driss"`), `Age` (21, 23, 20) et `Note` (14.5, 16.0, 12.0)
2. `plus_de_21` : le **nombre** d'étudiants de plus de 21 ans

<details>
<summary>💡 Indice 1 · la piste</summary>

Un dictionnaire : une **clé** par colonne, une **liste** de valeurs par clé.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.DataFrame({"Nom": [...], "Age": [...], "Note": [...]})` ; pour compter, un masque `promo["Age"] > 21` puis `.sum()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`plus_de_21 = (promo["Age"] > 21).sum()` : un seul étudiant, Sara.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
promo = ...
plus_de_21 = ...

verifier("A2", "promo est un DataFrame de forme (3, 3)", lambda: isinstance(promo, pd.DataFrame) and promo.shape == (3, 3))
verifier("A2", "les colonnes sont Nom, Age, Note", lambda: list(promo.columns) == ["Nom", "Age", "Note"])
verifier("A2", "un seul étudiant a plus de 21 ans", lambda: plus_de_21 == 1)

**✏️ Exercice A3 — Additionner par étiquette ⭐**

Avec `s1` et `s2` (cellule précédente) :
1. `somme` : `s1 + s2`
2. `nb_nan` : le **nombre de NaN** dans `somme`
3. `valeur_b` : la valeur de `somme` à l'étiquette `"b"`

<details>
<summary>💡 Indice 1 · la piste</summary>

Additionnez directement les deux Series ; les NaN se comptent avec `isna()` puis `sum()`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`somme = s1 + s2` ; `nb_nan = somme.isna().sum()` ; `valeur_b = somme["b"]`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Les étiquettes `a` et `d` n'ont pas de vis-à-vis : 2 NaN. `b` vaut 20 + 100 = 120.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
somme = ...
nb_nan = ...
valeur_b = ...

verifier("A3", "somme est une Series de 4 étiquettes", lambda: isinstance(somme, pd.Series) and list(somme.index) == ["a", "b", "c", "d"])
verifier("A3", "deux NaN, aux étiquettes a et d", lambda: nb_nan == 2 and somme.isna()["a"] and somme.isna()["d"])
verifier("A3", "b vaut 120", lambda: valeur_b == 120)

**✏️ Exercice A4 — Depuis une liste de dictionnaires ⭐**

En séance 1, `csv.DictReader` donnait une **liste de dictionnaires**. Avec `liste` ci-dessous :
1. `df_liste` : le **DataFrame** construit depuis `liste`
2. `tableau` : le **tableau NumPy** qui se cache dessous (`to_numpy`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`pd.DataFrame` accepte directement une liste de dictionnaires : un dictionnaire par ligne.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df_liste = pd.DataFrame(liste)` ; `tableau = df_liste.to_numpy()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Le DataFrame a 3 lignes et 2 colonnes (`Age`, `Classe`) ; le tableau a la même forme.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
liste = [{"Age": 22, "Classe": 3}, {"Age": 38, "Classe": 1}, {"Age": 26, "Classe": 3}]
df_liste = ...
tableau = ...

verifier("A4", "un DataFrame de forme (3, 2)", lambda: isinstance(df_liste, pd.DataFrame) and df_liste.shape == (3, 2))
verifier("A4", "colonnes Age et Classe", lambda: list(df_liste.columns) == ["Age", "Classe"])
verifier("A4", "tableau NumPy de forme (3, 2)", lambda: isinstance(tableau, np.ndarray) and tableau.shape == (3, 2))

**✏️ Exercice A5 — Ranger une colonne face à son étiquette ⭐⭐**

`notes` et `villes` sont deux Series qui **partagent les mêmes étiquettes** (des numéros d'inscrit), **dans un ordre différent**. `table` ne contient pour l'instant que les notes. **Ajoutez la colonne `Ville`** à `table`, prise dans `villes` : sans boucle, sans trier, et sans vous soucier de l'ordre.

<details>
<summary>💡 Indice 1 · la piste</summary>

L'**alignement par étiquette** (somme de deux Series, plus haut) joue aussi quand on affecte une colonne : chaque valeur se range face à son étiquette.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

Affectez une Series à une nouvelle colonne : `table["Ville"] = ...`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`table["Ville"] = villes` : le numéro 3 reçoit `"Khouribga"`, même s'il est en premier dans `table`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
notes = pd.Series([14.5, 16.0, 12.0], index=[3, 1, 2])
villes = pd.Series(["Fes", "Meknes", "Khouribga"], index=[1, 2, 3])
table = pd.DataFrame({"Note": notes})

# votre code : ajoutez la colonne Ville à table
...

print(table)

verifier("A5", "table a deux colonnes, Note puis Ville, et garde ses trois lignes", lambda: list(table.columns) == ["Note", "Ville"] and len(table) == 3)
verifier("A5", "les étiquettes restent dans l'ordre 3, 1, 2", lambda: "Ville" in table.columns and list(table.index) == [3, 1, 2])
verifier("A5", "chaque ville est face à son numéro : le 3 est de Khouribga, le 1 de Fes", lambda: table.loc[3, "Ville"] == "Khouribga" and table.loc[1, "Ville"] == "Fes" and table.loc[2, "Ville"] == "Meknes")

---
# Partie B · Charger, regarder, écrire ⭐⭐
_Diapos 11 à 15._

**Le fil rouge de la partie 1 : 20 inscrits d'une faculté.** Le fichier est **écrit par la cellule ci‑dessous** dans un dossier `fichiers_tp/` (créé à côté du notebook), puis **lu** avec `read_csv`. Il a des défauts exprès : des âges et des notes manquants, des villes écrites de plusieurs façons.

**Cours en bref**
- `pd.read_csv(chemin)` lit un fichier texte et rend un **DataFrame**. Paramètres utiles : `sep=";"` (séparateur), `decimal=","` (virgule décimale), `encoding="latin-1"` (encodage), `nrows=10` (seulement 10 lignes), `index_col="Id"` (une colonne devient l'**index**).
- **Un fichier abîmé se lit au symptôme** : une seule colonne → `sep` ; caractères absurdes → `encoding` ; nombres lus comme du texte → `decimal`.
- **Cinq gestes pour regarder** : `df.head(n)`, `df.tail(n)`, `df.shape` (lignes, colonnes), `df.columns`, `df.index`.
- `df.info()` donne, pour chaque colonne, le **nombre de valeurs présentes** et le **type** ; `df.dtypes` donne les types seuls.
- `df.describe()` résume les colonnes **numériques** (count, mean, std, min, quartiles, max) ; pour le texte, `df["col"].value_counts()`.
- `df.to_csv(chemin, index=False)` **écrit** un DataFrame ; sans `index=False`, l'index devient une première colonne du fichier.
- **Où sont mes fichiers ?** Le notebook crée le dossier `fichiers_tp/` à côté de lui ; `dossier.iterdir()` liste son contenu. **Dans Colab**, ce dossier est dans l'espace **temporaire** de la session : ouvrez le panneau **📁** à gauche, dossier `fichiers_tp` (bouton **↻** pour actualiser). Les fichiers **disparaissent** à la fermeture ; pour en garder un : clic droit dessus ▸ **Télécharger**.

In [ ]:
INSCRITS = """Id,Nom,Filiere,Ville,Age,Note,Bourse
1,Ali,IA,Fes,21,14.5,non
2,Sara,GI,Meknes,23,16.0,oui
3,Driss,IA,Fes,20,12.0,non
4,Salma,GI,Khouribga,22,,oui
5,Youssef,IA,fes ,24,9.5,non
6,Imane,GI,Meknes,,15.0,non
7,Hamza,IA,FES,21,11.0,oui
8,Nada,GI,Khouribga,25,17.5,non
9,Omar,IA,Fes,22,8.0,non
10,Lina,GI,Meknes,23,13.5,oui
11,Karim,IA,Khouribga,,10.0,non
12,Jalil,GI,fes,20,,non
13,Rim,IA,Meknes,24,12.5,oui
14,Anas,GI,FES,22,14.0,non
15,Hind,IA,Khouribga,21,16.5,oui
16,Zineb,GI,Meknes,23,15.5,non
17,Reda,IA,Fes,26,9.0,non
18,Meryem,GI,Khouribga,,13.0,oui
19,Tarik,IA,Meknes ,22,11.5,non
20,Aya,GI,Fes,21,18.0,oui
"""

dossier = Path("fichiers_tp")                       # le dossier d'essai, à côté du notebook
dossier.mkdir(exist_ok=True)
chemin = dossier / "inscrits.csv"
chemin.write_text(INSCRITS, encoding="utf-8")       # le fichier est écrit...

print("le fichier est ici :", chemin.resolve())      # son adresse complète (dans Colab : panneau 📁, dossier fichiers_tp)
print("le dossier contient :", sorted(p.name for p in dossier.iterdir()))

df = pd.read_csv(chemin)                            # ...puis lu : il devient un DataFrame
print("forme :", df.shape)                          # (lignes, colonnes)
print("colonnes :", list(df.columns))
df.head()                                           # les 5 premières lignes

In [ ]:
df.info()                                           # valeurs présentes et type de chaque colonne

In [ ]:
print(df[["Age", "Note"]].describe().round(2))
print(df["Filiere"].value_counts())                 # le texte se résume avec value_counts

**✏️ Exercice B1 — Lire un fichier ⭐**

Avec `chemin` (le fichier des inscrits) :
1. `cinq` : les **5 premières lignes** du fichier, lues avec `read_csv` (paramètre `nrows`)
2. `par_id` : le fichier complet relu avec **la colonne `Id` comme index** (paramètre `index_col`)
3. `nb_lignes` et `nb_colonnes` : la forme du DataFrame `df`

<details>
<summary>💡 Indice 1 · la piste</summary>

`read_csv` accepte un paramètre qui limite le nombre de lignes lues, et un autre qui choisit la colonne d'index.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.read_csv(chemin, nrows=5)` ; `pd.read_csv(chemin, index_col="Id")` ; `df.shape` renvoie un couple `(lignes, colonnes)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`cinq = pd.read_csv(chemin, nrows=5)` ; `par_id = pd.read_csv(chemin, index_col="Id")` ; `nb_lignes, nb_colonnes = df.shape`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
cinq = ...
par_id = ...
nb_lignes, nb_colonnes = ..., ...

verifier("B1", "cinq est un DataFrame de 5 lignes et 7 colonnes", lambda: isinstance(cinq, pd.DataFrame) and cinq.shape == (5, 7))
verifier("B1", "par_id a l'index Id (de 1 à 20) et 6 colonnes", lambda: par_id.index.name == "Id" and list(par_id.index) == list(range(1, 21)) and par_id.shape == (20, 6))
verifier("B1", "20 inscrits, 7 colonnes", lambda: nb_lignes == 20 and nb_colonnes == 7)

**✏️ Exercice B2 — Réparer un fichier abîmé ⭐⭐**

Un tableur installé en français a exporté ce fichier : séparateur **point-virgule**, virgule **décimale**.
`texte_csv` est le contenu du fichier (on le lit avec `io.StringIO`). Lisez-le correctement :
1. `notes_df` : le DataFrame lu avec les **bons paramètres** (`sep` et `decimal`), la colonne `Nom` comme index
2. `moyenne_notes` : la moyenne de la colonne `Note`

<details>
<summary>💡 Indice 1 · la piste</summary>

Sans paramètre, tout tient dans **une seule colonne**. Trois paramètres réparent le fichier et rangent les noms en index.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`pd.read_csv(io.StringIO(texte_csv), sep=";", decimal=",", index_col="Nom")`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`notes_df = pd.read_csv(io.StringIO(texte_csv), sep=";", decimal=",", index_col="Nom")` ; `moyenne_notes = notes_df["Note"].mean()`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
texte_csv = "Nom;Age;Note\nAli;21;14,5\nSara;23;16,0\nDriss;20;12,0"
notes_df = ...
moyenne_notes = ...

verifier("B2", "deux colonnes (Age, Note), trois lignes, index Nom", lambda: isinstance(notes_df, pd.DataFrame) and notes_df.shape == (3, 2) and notes_df.index.name == "Nom")
verifier("B2", "la colonne Note est numérique", lambda: pd.api.types.is_float_dtype(notes_df["Note"]))
verifier("B2", "moyenne des notes : 14,17", lambda: abs(moyenne_notes - 14.1667) < 0.001)

**✏️ Exercice B3 — Feuilleter un DataFrame ⭐**

Sur `df` :
1. `trois_premiers` : les **3 premières lignes**
2. `index_fin` : la **liste des étiquettes** des 5 dernières lignes (`tail`)
3. `noms_colonnes` : la **liste des noms de colonnes**

<details>
<summary>💡 Indice 1 · la piste</summary>

`head(n)` et `tail(n)` ; l'index d'un DataFrame se lit avec `.index`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df.head(3)` ; `df.tail(5).index.tolist()` ; `list(df.columns)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`trois_premiers = df.head(3)` ; `index_fin = df.tail(5).index.tolist()` ; `noms_colonnes = list(df.columns)`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
trois_premiers = ...
index_fin = ...
noms_colonnes = ...

verifier("B3", "trois lignes", lambda: isinstance(trois_premiers, pd.DataFrame) and len(trois_premiers) == 3)
verifier("B3", "les 5 dernières étiquettes : 15 à 19", lambda: list(index_fin) == [15, 16, 17, 18, 19])
verifier("B3", "7 noms de colonnes, le premier est Id", lambda: list(noms_colonnes)[0] == "Id" and len(noms_colonnes) == 7)

**✏️ Exercice B4 — Lire info et describe ⭐⭐**

Sans relire la sortie de `info()` ni de `describe()` à l'œil, **calculez** :
1. `nb_ages` : le nombre de valeurs **présentes** dans `Age` (`count`)
2. `moyenne_note` et `mediane_note` : la moyenne et la médiane de `Note`
3. `nb_ia` : le nombre d'inscrits de la filière `IA` (`value_counts` sur `Filiere`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`count()` compte les valeurs présentes ; `mean()` et `median()` donnent moyenne et médiane.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df["Age"].count()` ; `df["Note"].mean()` ; `df["Note"].median()` ; `df["Filiere"].value_counts()["IA"]`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`nb_ages = df["Age"].count()` ; `nb_ia = df["Filiere"].value_counts()["IA"]`. Les notes manquantes sont ignorées par `mean` et `median`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
nb_ages = ...
moyenne_note = ...
mediane_note = ...
nb_ia = ...

verifier("B4", "17 âges présents sur 20", lambda: nb_ages == 17)
verifier("B4", "moyenne des notes : 13,17", lambda: abs(moyenne_note - 13.1667) < 0.01)
verifier("B4", "médiane des notes : 13,25", lambda: abs(mediane_note - 13.25) < 0.01)
verifier("B4", "10 inscrits en IA", lambda: nb_ia == 10)

**✏️ Exercice B5 — Écrire un fichier ⭐⭐**

`sortie` est le chemin du fichier à écrire (fourni). Écrivez-y **seulement les inscrits de la filière `IA`**, **sans la colonne d'index** :
1. `ia` : le DataFrame des inscrits de la filière `IA`
2. Écrivez `ia` dans `sortie` avec `to_csv`

<details>
<summary>💡 Indice 1 · la piste</summary>

Un masque garde les lignes voulues ; `to_csv` est l'écriture de `read_csv`. Sans précaution, l'index est écrit comme une colonne de plus.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`ia = df[df["Filiere"] == "IA"]` ; `ia.to_csv(sortie, index=False)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Le fichier relu a 10 lignes et 7 colonnes : les mêmes que `df`, sans colonne d'index en plus.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
sortie = dossier / "inscrits_ia.csv"
sortie.unlink(missing_ok=True)                      # le fichier n'existe pas encore

ia = ...
# votre code : écrivez ia dans sortie avec to_csv

verifier("B5", "ia : 10 inscrits, tous en IA", lambda: isinstance(ia, pd.DataFrame) and len(ia) == 10 and (ia["Filiere"] == "IA").all())
verifier("B5", "le fichier existe et se relit avec 10 lignes et 7 colonnes (sans colonne d'index en plus)", lambda: pd.read_csv(sortie).shape == (10, 7))
verifier("B5", "la première colonne du fichier est Id", lambda: list(pd.read_csv(sortie).columns)[0] == "Id")

---
# Partie C · Sélectionner ⭐⭐
_Diapos 16 à 21._

**Cours en bref**
- `df["col"]` → une **Series** ; `df[["col1", "col2"]]` → un **DataFrame** (le crochet intérieur est la liste de noms).
- `df.loc[etiquette_ligne, "colonne"]` lit **par étiquette** ; `df.iloc[ligne, colonne]` lit **par position** (à partir de 0). Avec l'index par défaut, étiquette et position **coïncident** : c'est le piège. Ici, `d` a le numéro d'inscrit `Id` pour index (de 1 à 20) : **étiquette et position diffèrent**.
- `d.columns.get_loc("Age")` donne la **position** d'une colonne, pour la lire avec `iloc`.
- **Tranches** : `loc[1:3]` **inclut** la borne de fin (3 lignes) ; `iloc[1:3]` l'**exclut** (2 lignes).
- **Masque** : `df["Age"] < 22` est une Series de vrai et faux ; `df[masque]` garde les lignes vraies. Un âge **NaN** donne `False` : la ligne n'est pas gardée.
- **Combiner** : `&` (et), `|` (ou), `~` (non), **chaque condition entre parenthèses** ; `isin(["Fes", "Meknes"])` teste l'appartenance à une liste.

In [ ]:
note = df["Note"]                       # une Series
deux = df[["Nom", "Note"]]              # un DataFrame
print(type(note).__name__, note.shape, "·", type(deux).__name__, deux.shape)

d = df.set_index("Id")                  # l'index devient le numéro d'inscrit : 1, 2, 3...
print("loc  :", d.loc[1, "Nom"], "·", "iloc :", d.iloc[0, d.columns.get_loc("Nom")])
print("loc[1:3] :", len(d.loc[1:3]), "lignes · iloc[1:3] :", len(d.iloc[1:3]), "lignes")

In [ ]:
masque = d["Age"] < 22                  # vrai pour les moins de 22 ans, faux sinon (et pour les NaN)
print("moins de 22 ans :", masque.sum())
jeunes = d[masque]                      # on garde les lignes où le masque est vrai
print(jeunes[["Nom", "Age", "Note"]].head(3))

ia_boursiers = d[(d["Filiere"] == "IA") & (d["Bourse"] == "oui")]      # chaque condition entre parenthèses
print("IA avec bourse :", len(ia_boursiers))

**✏️ Exercice C1 — Une colonne ou plusieurs ⭐**

1. `col_note` : la colonne `Note` (une **Series**)
2. `deux_colonnes` : un **DataFrame** avec seulement `Nom` et `Filiere`

<details>
<summary>💡 Indice 1 · la piste</summary>

Un seul nom entre crochets, ou une **liste** de noms entre doubles crochets.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df["Note"]` et `df[["Nom", "Filiere"]]`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`col_note = df["Note"]` ; `deux_colonnes = df[["Nom", "Filiere"]]`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
col_note = ...
deux_colonnes = ...

verifier("C1", "col_note est une Series de 20 valeurs", lambda: isinstance(col_note, pd.Series) and col_note.shape == (20,))
verifier("C1", "deux_colonnes est un DataFrame de forme (20, 2)", lambda: isinstance(deux_colonnes, pd.DataFrame) and deux_colonnes.shape == (20, 2))

**✏️ Exercice C2 — loc ou iloc ⭐⭐**

Avec `d` (index = numéro d'inscrit) :
1. `age_loc` : l'âge de l'inscrit numéro **5**, lu **par étiquette**
2. `age_iloc` : le même âge, lu **par position** (l'inscrit 5 est à la position 4)

<details>
<summary>💡 Indice 1 · la piste</summary>

`loc` prend des étiquettes ; `iloc` des positions. La colonne `Age` : `d.columns.get_loc("Age")` donne sa position.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`d.loc[5, "Age"]` ; `d.iloc[4, d.columns.get_loc("Age")]`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Les deux valent 24.0 : c'est la **même case**.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
age_loc = ...
age_iloc = ...

verifier("C2", "l'âge de l'inscrit 5 vaut 24 (loc)", lambda: age_loc == 24.0)
verifier("C2", "l'âge de l'inscrit 5 vaut 24 (iloc)", lambda: age_iloc == 24.0)

**✏️ Exercice C3 — Tranche incluse, tranche exclue ⭐⭐**

Avec `d` :
1. `n_loc` : le nombre de lignes de `d.loc[1:5]`
2. `n_iloc` : le nombre de lignes de `d.iloc[1:5]`
3. `etiquettes_iloc` : la **liste des étiquettes** des lignes de `d.iloc[1:5]`

<details>
<summary>💡 Indice 1 · la piste</summary>

La borne de fin est **incluse** avec `loc`, **exclue** avec `iloc`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`len(d.loc[1:5])` ; `len(d.iloc[1:5])` ; `d.iloc[1:5].index.tolist()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`loc[1:5]` : étiquettes 1 à 5, soit 5 lignes. `iloc[1:5]` : positions 1 à 4, soit 4 lignes, d'étiquettes 2 à 5.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
n_loc = ...
n_iloc = ...
etiquettes_iloc = ...

verifier("C3", "loc[1:5] garde 5 lignes", lambda: n_loc == 5)
verifier("C3", "iloc[1:5] garde 4 lignes", lambda: n_iloc == 4)
verifier("C3", "les étiquettes de iloc[1:5] : 2 à 5", lambda: list(etiquettes_iloc) == [2, 3, 4, 5])

**✏️ Exercice C4 — Un masque ⭐⭐**

1. `bonnes_notes` : le sous-tableau des inscrits dont la **note est d'au moins 15**
2. `nb_bonnes_notes` : leur nombre
3. `nb_sans_age_parmi_jeunes` : le nombre d'inscrits **sans âge** parmi ceux qui ont **moins de 22 ans** (réponse attendue : pourquoi ?)

<details>
<summary>💡 Indice 1 · la piste</summary>

`d[condition]` garde les lignes vraies ; `len` compte.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`d[d["Note"] >= 15]` ; pour la 3e, filtrez `d["Age"] < 22` puis comptez les `isna()` de `Age`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`nb_sans_age_parmi_jeunes = d[d["Age"] < 22]["Age"].isna().sum()` vaut **0** : un NaN compare toujours faux, la ligne est déjà éliminée.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
bonnes_notes = ...
nb_bonnes_notes = ...
nb_sans_age_parmi_jeunes = ...

verifier("C4", "bonnes_notes est un DataFrame dont toutes les notes sont d'au moins 15", lambda: isinstance(bonnes_notes, pd.DataFrame) and (bonnes_notes["Note"] >= 15).all())
verifier("C4", "6 inscrits ont au moins 15", lambda: nb_bonnes_notes == 6)
verifier("C4", "aucun inscrit sans âge parmi les moins de 22 ans", lambda: nb_sans_age_parmi_jeunes == 0)

**✏️ Exercice C5 — Combiner des conditions ⭐⭐**

1. `ia_boursiers` : le sous-tableau des inscrits de la filière **IA** qui ont une **bourse** (`Bourse` vaut `"oui"`)
2. `n_fes_ou_meknes` : le nombre d'inscrits dont la ville est **exactement** `"Fes"` ou `"Meknes"` (avec `isin`)
3. `n_sans_bourse` : le nombre d'inscrits qui **n'ont pas** de bourse (avec `~`)

<details>
<summary>💡 Indice 1 · la piste</summary>

Chaque condition entre parenthèses ; `&` pour et ; `~` pour non ; `isin` pour une liste de valeurs.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`d[(d["Filiere"] == "IA") & (d["Bourse"] == "oui")]` ; `d["Ville"].isin(["Fes", "Meknes"]).sum()` ; `(~(d["Bourse"] == "oui")).sum()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Les valeurs attendues : 3 ; 10 (les villes écrites `"fes "` ou `"FES"` ne comptent pas : on y revient en partie E) ; 12.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
ia_boursiers = ...
n_fes_ou_meknes = ...
n_sans_bourse = ...

verifier("C5", "3 inscrits en IA avec bourse", lambda: isinstance(ia_boursiers, pd.DataFrame) and len(ia_boursiers) == 3)
verifier("C5", "10 inscrits à Fes ou Meknes, orthographe exacte", lambda: n_fes_ou_meknes == 10)
verifier("C5", "12 inscrits sans bourse", lambda: n_sans_bourse == 12)

**✏️ Exercice C6 — À vous : les boursiers de la filière IA ⭐⭐⭐**

`part_boursiers_ia` : la **proportion** d'inscrits qui ont une bourse parmi ceux de la filière **IA**. Un test (`== "oui"`) donne vrai ou faux : la moyenne de vrai et faux est la **proportion** de vrai.

<details>
<summary>💡 Indice 1 · la piste</summary>

Trois pièces : un masque, une colonne, une moyenne.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`d[d["Filiere"] == "IA"]["Bourse"]` donne les bourses des inscrits en IA ; comparez-les à `"oui"`, puis `mean()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`part_boursiers_ia = (d[d["Filiere"] == "IA"]["Bourse"] == "oui").mean()` : 3 sur 10, soit 0,3.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
part_boursiers_ia = ...

verifier("C6", "30 % des inscrits en IA ont une bourse", lambda: abs(part_boursiers_ia - 0.3) < 0.001)

---
# Partie D · Diagnostiquer ⭐⭐
_Diapos 22 à 25._

**Cours en bref**
- **Six gestes, dans cet ordre, sur tout jeu inconnu** : `shape`, `head`, `dtypes`, `info`, `isna().sum()`, `value_counts()`. Le cinquième est le plus important : un trou ignoré fausse les calculs qui suivent.
- `df.isna()` marque chaque case vide (`True`), `sum()` les compte par colonne ; la **moyenne** d'un tableau de vrai et faux est la **part** de vrai : `df.isna().mean()` donne la part de trous.
- `df.duplicated()` marque les lignes en **double** (la première occurrence est gardée) ; `drop_duplicates()` les retire.
- `value_counts()` révèle les **libellés incohérents** : majuscule, espace en trop, abréviation ; `nunique()` compte les **valeurs distinctes** d'une colonne.

In [ ]:
trous = df.isna().sum()
print(trous[trous > 0])                          # seules les colonnes incomplètes
print((df.isna().mean() * 100).round(1)[["Age", "Note"]])

petit = pd.DataFrame({"Nom": ["Ali", "Sara", "Ali", "Driss", "Sara"], "Age": [21, 23, 21, 20, 23], "Note": [14.5, 16.0, 14.5, 12.0, 16.0]})
print(petit.duplicated().tolist())               # True pour chaque répétition
print(petit.drop_duplicates().shape, "· doublons dans les inscrits :", df.duplicated().sum())

**✏️ Exercice D1 — Compter les trous ⭐⭐**

1. `nb_trous` : le nombre de valeurs manquantes **par colonne** (une Series)
2. `colonnes_incompletes` : la **liste des colonnes** qui ont au moins un trou, dans l'ordre du fichier

<details>
<summary>💡 Indice 1 · la piste</summary>

`isna()` marque les trous, `sum()` les compte par colonne ; on filtre ensuite les colonnes dont le compte dépasse 0.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`nb_trous = df.isna().sum()` ; `colonnes_incompletes = list(nb_trous[nb_trous > 0].index)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Deux colonnes : `Age` (3 trous) et `Note` (2 trous).

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
nb_trous = ...
colonnes_incompletes = ...

verifier("D1", "nb_trous est une Series de 7 valeurs (une par colonne)", lambda: isinstance(nb_trous, pd.Series) and len(nb_trous) == 7)
verifier("D1", "3 âges manquants et 2 notes manquantes", lambda: nb_trous["Age"] == 3 and nb_trous["Note"] == 2)
verifier("D1", "les colonnes incomplètes : Age et Note", lambda: list(colonnes_incompletes) == ["Age", "Note"])

**✏️ Exercice D2 — La part de trous ⭐⭐**

1. `part_age` : la **part** de valeurs manquantes dans `Age` (un nombre entre 0 et 1)
2. `part_note` : la part de valeurs manquantes dans `Note`
3. `a_traiter` : `True` si la colonne `Age` a **plus de 10 %** de trous, sinon `False`

<details>
<summary>💡 Indice 1 · la piste</summary>

La moyenne d'un tableau de vrai et faux est la part de vrai.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df["Age"].isna().mean()` ; `df["Note"].isna().mean()` ; `a_traiter = part_age > 0.10`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`part_age` vaut 3 sur 20, soit 0,15 ; `part_note` vaut 0,10 ; `a_traiter` est `True`.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
part_age = ...
part_note = ...
a_traiter = ...

verifier("D2", "15 % des âges manquent", lambda: abs(part_age - 0.15) < 0.001)
verifier("D2", "10 % des notes manquent", lambda: abs(part_note - 0.10) < 0.001)
verifier("D2", "Age dépasse 10 % de trous", lambda: isinstance(a_traiter, (bool, np.bool_)) and bool(a_traiter) is True)

**✏️ Exercice D3 — Doublons ⭐⭐**

`petit` (cellule du cours) contient des lignes répétées. Calculez :
1. `nb_doublons` : le nombre de lignes en **double** (à compter à partir de la 2ᵉ occurrence)
2. `sans_doublon` : `petit` **sans** les doublons

<details>
<summary>💡 Indice 1 · la piste</summary>

`duplicated()` marque chaque répétition ; `drop_duplicates()` les retire en gardant la première.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`petit.duplicated().sum()` ; `petit.drop_duplicates()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Deux répétitions (la 2ᵉ Ali, la 2ᵉ Sara) : il reste 3 lignes.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
nb_doublons = ...
sans_doublon = ...

verifier("D3", "2 lignes en double", lambda: nb_doublons == 2)
verifier("D3", "sans_doublon : 3 lignes, 3 colonnes", lambda: isinstance(sans_doublon, pd.DataFrame) and sans_doublon.shape == (3, 3))
verifier("D3", "les doublons de petit sont comptés, et les inscrits n'en ont aucun", lambda: nb_doublons == 2 and df.duplicated().sum() == 0)

**✏️ Exercice D4 — Repérer les libellés incohérents ⭐⭐**

La colonne `Ville` est écrite de plusieurs façons. Avec `value_counts()` :
1. `nb_libelles` : le nombre de **libellés distincts** dans `df["Ville"]`
2. `nb_villes` : le nombre de **villes réelles**, une fois les espaces retirés (`.str.strip()`) et les majuscules passées en minuscules (`.str.lower()`)

<details>
<summary>💡 Indice 1 · la piste</summary>

`nunique()` compte les valeurs distinctes. Le texte d'une colonne se transforme avec `.str.` : `.str.strip()` retire les espaces aux extrémités, `.str.lower()` passe en minuscules.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df["Ville"].nunique()` ; `df["Ville"].str.strip().str.lower().nunique()`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`Fes`, `fes `, `FES` et `fes` désignent la même ville, comme `Meknes` et `Meknes ` : 7 libellés, 3 villes.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
nb_libelles = ...
nb_villes = ...

verifier("D4", "7 libellés distincts", lambda: nb_libelles == 7)
verifier("D4", "3 villes réelles", lambda: nb_villes == 3)

---
# Partie E · Nettoyer ⭐⭐⭐
_Diapos 26 à 32._

**Cours en bref**
- **Nettoyer, ce n'est pas réparer, c'est décider** : supprimer, remplir, marquer ou laisser. On travaille sur une **copie** (`df.copy()`) pour garder l'original.
- **Supprimer** : `dropna()` retire toute ligne ayant un trou (ici 20 → 15) ; `dropna(subset=["Age"])` ne regarde que `Age` (20 → 17). **On annonce le coût avant de supprimer.**
- **Remplir** : `df["Age"] = df["Age"].fillna(valeur)`. La médiane résiste mieux que la moyenne aux valeurs extrêmes, mais empile les âges inventés sur une seule valeur. Pour du **texte**, on remplit par le **mode**, la valeur la plus fréquente : `df["Ville"].mode()[0]`. **Pas d'`inplace=True` sur une colonne** : en Pandas 3, le chaînage ne modifie plus le DataFrame.
- **Marquer** d'abord, remplir ensuite : `df["Age_estime"] = df["Age"].isna()`.
- **Libellés** : `.str.strip()`, `.str.lower()`, `replace({...})`.
- **Types** : `astype("category")` range chaque libellé une fois (`.cat.categories` les liste) ; `astype(int)` **échoue** sur un NaN (remplir d'abord).
- **Colonnes** : `df["Note_sur_100"] = df["Note"] * 5` (calcul vectorisé, sans boucle), `rename(columns={...})`, `sort_values("col", ascending=False)`.

In [ ]:
print("lignes :", len(df), "· dropna() :", len(df.dropna()), "· dropna(subset=['Age']) :", len(df.dropna(subset=["Age"])))

copie = df.copy()                                          # on nettoie une copie
mediane = copie["Age"].median()
copie["Age_estime"] = copie["Age"].isna()                  # 1. marquer
copie["Age"] = copie["Age"].fillna(mediane)                # 2. remplir
print("médiane :", mediane, "· âges estimés :", copie["Age_estime"].sum(), "· trous restants :", copie["Age"].isna().sum())

copie["Note_sur_100"] = copie["Note"] * 5                  # une colonne calculée, sans boucle
print(copie[["Nom", "Note", "Note_sur_100"]].head(3))

try:
    df["Age"].astype(int)                                  # un entier ne peut pas être NaN
except Exception as e:
    print("astype(int) sur des âges manquants :", type(e).__name__)

**✏️ Exercice E1 — Annoncer le coût d'une suppression ⭐⭐**

1. `n_sans_trou` : le nombre de lignes qui restent avec `dropna()`
2. `n_sans_age` : le nombre de lignes qui restent avec `dropna(subset=["Age"])`
3. `part_perdue` : la **part des 20 inscrits perdue** avec `dropna()` (un nombre entre 0 et 1)

<details>
<summary>💡 Indice 1 · la piste</summary>

`dropna()` regarde toutes les colonnes ; avec `subset=[...]`, seulement celles de la liste. Le coût se mesure par le nombre de lignes avant et après.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`len(df.dropna())` ; `len(df.dropna(subset=["Age"]))` ; `part_perdue = 1 - n_sans_trou / len(df)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Avec `dropna()`, il reste 15 lignes sur 20 : un quart des inscrits disparaît. Avec `subset=["Age"]`, il en reste 17.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
n_sans_trou = ...
n_sans_age = ...
part_perdue = ...

verifier("E1", "dropna() garde 15 lignes", lambda: n_sans_trou == 15)
verifier("E1", "dropna(subset=['Age']) garde 17 lignes", lambda: n_sans_age == 17)
verifier("E1", "un quart des inscrits perdu", lambda: abs(part_perdue - 0.25) < 0.001)

**✏️ Exercice E2 — Remplir par la médiane ⭐⭐**

Sur une **copie** de `df` :
1. `copie_age` : la copie, dont la colonne `Age` est **remplie par la médiane**
2. `mediane_age` : la valeur de cette médiane

<details>
<summary>💡 Indice 1 · la piste</summary>

On calcule la médiane **avant** de remplir ; `fillna(valeur)` remplace les trous. Pas d'`inplace`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`mediane_age = df["Age"].median()` ; `copie_age = df.copy()` ; `copie_age["Age"] = copie_age["Age"].fillna(mediane_age)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

La médiane vaut 22.0 : les 3 âges manquants deviennent 22.0 ; `df` est inchangé.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
mediane_age = ...
copie_age = ...

verifier("E2", "la médiane des âges vaut 22", lambda: mediane_age == 22.0)
verifier("E2", "copie_age n'a plus aucun trou dans Age, et df est inchangé (3 âges manquants)", lambda: isinstance(copie_age, pd.DataFrame) and copie_age["Age"].isna().sum() == 0 and df["Age"].isna().sum() == 3)
verifier("E2", "les 3 âges manquants valent maintenant la médiane (22)", lambda: isinstance(copie_age, pd.DataFrame) and (copie_age.loc[df["Age"].isna(), "Age"] == 22.0).all())

**✏️ Exercice E3 — Marquer avant de remplir ⭐⭐**

Sur une **copie** de `df`, dans cet ordre :
1. `marquee` : la copie
2. `marquee["Age_estime"]` : `True` là où `Age` est **manquant**
3. puis `Age` **rempli** par la médiane

<details>
<summary>💡 Indice 1 · la piste</summary>

Marquer d'abord, remplir ensuite : une fois les trous remplis, on ne sait plus où ils étaient.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`marquee = df.copy()` ; `marquee["Age_estime"] = marquee["Age"].isna()` ; puis le `fillna` de l'exercice précédent.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

3 âges marqués estimés : les inscrits 6, 11 et 18.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
marquee = ...

verifier("E3", "Age_estime existe et compte 3 âges estimés", lambda: isinstance(marquee, pd.DataFrame) and "Age_estime" in marquee.columns and marquee["Age_estime"].sum() == 3)
verifier("E3", "Age n'a plus de trou", lambda: marquee["Age"].isna().sum() == 0)
verifier("E3", "les âges estimés valent tous la médiane (22)", lambda: (marquee.loc[marquee["Age_estime"], "Age"] == 22.0).all())

**✏️ Exercice E4 — Uniformiser des libellés ⭐⭐⭐**

1. `ville_propre` : la colonne `Ville` **sans espace aux extrémités** et **en minuscules**
2. `effectifs_villes` : le nombre d'inscrits **par ville propre** (`value_counts`)

<details>
<summary>💡 Indice 1 · la piste</summary>

Deux méthodes `.str` à enchaîner, puis `value_counts`.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df["Ville"].str.strip().str.lower()` ; puis `.value_counts()` sur le résultat.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Trois villes : `fes` (9 inscrits), `meknes` (6) et `khouribga` (5).

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
ville_propre = ...
effectifs_villes = ...

verifier("E4", "ville_propre : trois valeurs distinctes", lambda: isinstance(ville_propre, pd.Series) and sorted(ville_propre.unique()) == ["fes", "khouribga", "meknes"])
verifier("E4", "9 inscrits à Fes, 6 à Meknes, 5 à Khouribga", lambda: effectifs_villes["fes"] == 9 and effectifs_villes["meknes"] == 6 and effectifs_villes["khouribga"] == 5)

**✏️ Exercice E5 — Changer de type ⭐⭐⭐**

1. `filiere_cat` : la colonne `Filiere` convertie en type **`category`**
2. `ages_entiers` : `Age` **rempli par la médiane**, puis converti en **entiers** (sans cela, `astype(int)` échoue : voir la cellule du cours)

<details>
<summary>💡 Indice 1 · la piste</summary>

`astype("category")` range chaque libellé une fois. Pour les entiers : remplissez d'abord, convertissez ensuite.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`df["Filiere"].astype("category")` ; `df["Age"].fillna(df["Age"].median()).astype(int)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

`filiere_cat.cat.categories` donne `GI` et `IA` ; la somme des âges entiers vaut 446.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
filiere_cat = ...
ages_entiers = ...

verifier("E5", "filiere_cat est de type category, avec deux catégories GI et IA", lambda: str(filiere_cat.dtype) == "category" and list(filiere_cat.cat.categories) == ["GI", "IA"])
verifier("E5", "ages_entiers : 20 entiers sans trou", lambda: np.issubdtype(ages_entiers.dtype, np.integer) and len(ages_entiers) == 20)
verifier("E5", "la somme des âges entiers vaut 446", lambda: ages_entiers.sum() == 446)

**✏️ Exercice E6 — Créer, renommer, trier ⭐⭐⭐**

Sur une **copie** `classement` de `df` :
1. créez la colonne **`Note_sur_100`** : la note multipliée par 5 (sans boucle)
2. **renommez** la colonne `Note` en `Moyenne`
3. **triez** `classement` par `Moyenne`, de la **plus grande à la plus petite** (gardez le résultat dans `classement`)

<details>
<summary>💡 Indice 1 · la piste</summary>

Une colonne calculée se crée comme une affectation ; `rename` prend un dictionnaire ancien nom → nouveau nom ; `sort_values` rend un **nouveau** DataFrame trié, à garder.

</details>

<details>
<summary>💡 Indice 2 · la méthode</summary>

`classement["Note_sur_100"] = classement["Note"] * 5` ; `classement = classement.rename(columns={"Note": "Moyenne"})` ; `classement = classement.sort_values("Moyenne", ascending=False)`.

</details>

<details>
<summary>💡 Indice 3 · presque la réponse</summary>

Le premier est Aya (18.0, soit 90 sur 100), puis Nada (17.5) et Hind (16.5). Les notes manquantes se placent à la fin.

</details>

_Bloqué ? Essayez les indices dans l'ordre._

In [ ]:
classement = df.copy()
# votre code : créez Note_sur_100, renommez Note en Moyenne, puis triez
...

verifier("E6", "Note_sur_100 existe : Ali (14,5) vaut 72,5", lambda: "Note_sur_100" in classement.columns and classement.loc[classement["Nom"] == "Ali", "Note_sur_100"].iloc[0] == 72.5)
verifier("E6", "Note est renommée Moyenne", lambda: "Moyenne" in classement.columns and "Note" not in classement.columns)
verifier("E6", "classement trié : Aya, Nada, Hind en tête", lambda: list(classement["Nom"].head(3)) == ["Aya", "Nada", "Hind"])

---
## 🎉 Partie 1 terminée
Vous savez maintenant **créer, charger, regarder, sélectionner, diagnostiquer et nettoyer** un tableau avec Pandas, et **écrire** le résultat. Le **mini‑projet** (notebook suivant) refait ces gestes sur le **Titanic**, avec le fichier que vous avez produit en séance 2.

## 📚 Pour aller plus loin
- Guide de l'utilisateur Pandas : https://pandas.pydata.org/docs/user_guide/index.html
- Valeurs manquantes : https://pandas.pydata.org/docs/user_guide/missing_data.html
- Notes de version de Pandas 3.0 (copy-on-write, type texte `str`) : https://pandas.pydata.org/docs/whatsnew/v3.0.0.html
- Lire des fichiers (`read_csv`, `read_excel`) : https://pandas.pydata.org/docs/user_guide/io.html